# Day 8 — KV Cache: the Core Inference Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day08-kv-cache.ipynb)

Every token you generate recomputes the keys and values of every token before it — unless you cache them. Today you add a KV cache to a tiny grouped-query-attention GPT, prove the outputs are identical, measure the speedup, and do the byte math for Llama-3.1-8B and 70B at real serving sizes.

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
# Expected: installs quietly, no errors.

## 1. A tiny GQA GPT with `use_cache`

Four layers, four query heads, two KV heads, d_model = 128 (head_dim = 32). The attention `forward` takes an optional `past` tuple of `(K, V)` and returns `present` covering all tokens seen. When `past` is given, new K/V are concatenated — the append-only log from the packet.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

class GQACausalAttn(nn.Module):
    """Grouped-query causal attention with KV-cache support."""
    def __init__(self, d_model=128, n_head=4, n_kv=2):
        super().__init__()
        assert n_head % n_kv == 0
        self.n_head, self.n_kv = n_head, n_kv
        self.d = d_model // n_head
        self.q = nn.Linear(d_model, n_head * self.d, bias=False)
        self.k = nn.Linear(d_model, n_kv * self.d, bias=False)
        self.v = nn.Linear(d_model, n_kv * self.d, bias=False)
        self.o = nn.Linear(n_head * self.d, d_model, bias=False)

    def forward(self, x, past=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.n_head, self.d).transpose(1, 2)
        k = self.k(x).view(B, T, self.n_kv, self.d).transpose(1, 2)
        v = self.v(x).view(B, T, self.n_kv, self.d).transpose(1, 2)
        if past is not None:                       # append to the cache log
            pk, pv = past
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)
        present = (k, v)
        rep = self.n_head // self.n_kv              # GQA: repeat KV heads
        k = k.repeat_interleave(rep, dim=1)
        v = v.repeat_interleave(rep, dim=1)
        Tq, Tk = q.shape[2], k.shape[2]
        off = Tk - Tq                               # tokens already cached
        att = (q @ k.transpose(-2, -1)) / self.d ** 0.5
        i = torch.arange(Tq, device=x.device)
        j = torch.arange(Tk, device=x.device)
        mask = j[None, :] > off + i[:, None]        # query i sees keys <= off+i
        att = att.masked_fill(mask, float("-inf"))
        y = (F.softmax(att, dim=-1) @ v).transpose(1, 2)
        y = y.reshape(B, T, self.n_head * self.d)
        return self.o(y), present


class Block(nn.Module):
    def __init__(self, d_model=128, n_head=4, n_kv=2):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.attn = GQACausalAttn(d_model, n_head, n_kv)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model),
                                 nn.GELU(), nn.Linear(4 * d_model, d_model))

    def forward(self, x, past=None):
        y, present = self.attn(self.ln1(x), past)
        x = x + y
        return x + self.mlp(self.ln2(x)), present


class MiniGPT(nn.Module):
    def __init__(self, vocab=64, d_model=128, n_layer=4,
                 n_head=4, n_kv=2, max_seq=512):
        super().__init__()
        self.tok = nn.Embedding(vocab, d_model)
        self.pos = nn.Embedding(max_seq, d_model)
        self.blocks = nn.ModuleList(
            [Block(d_model, n_head, n_kv) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab, bias=False)

    def forward(self, idx, past_kv=None):
        B, T = idx.shape
        off = past_kv[0][0].shape[2] if past_kv else 0   # absolute positions
        x = self.tok(idx) + self.pos(
            torch.arange(off, off + T, device=idx.device))[None]
        presents = []
        for blk, past in zip(self.blocks,
                               past_kv or [None] * len(self.blocks)):
            x, pr = blk(x, past)
            presents.append(pr)
        return self.head(self.ln_f(x)), presents


model = MiniGPT().eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"MiniGPT: {n_params/1e6:.2f}M params, 4 layers, 4Q/2KV heads")
# Expected: MiniGPT: 0.81M params, 4 layers, 4Q/2KV heads

## 2. Correctness first: cached decode must be bit-identical

Two generate modes, same seed. Cached mode feeds only the last token plus the cache after the first step; naive mode re-feeds the whole sequence every step (recomputing everything — the waste we're killing). The assert is the whole day in one line.

In [ ]:
@torch.no_grad()
def generate(model, idx, n_new, use_cache=True, seed=0):
    torch.manual_seed(seed)
    ids, past = idx, None
    for _ in range(n_new):
        x = ids[:, -1:] if (use_cache and past is not None) else ids
        logits, past = model(x, past if use_cache else None)
        nxt = logits[:, -1, :].argmax(-1, keepdim=True)
        ids = torch.cat([ids, nxt], dim=1)
    return ids


torch.manual_seed(7)
prompt = torch.randint(0, 64, (1, 10))
ids_cached = generate(model, prompt, 200, use_cache=True, seed=42)
ids_plain = generate(model, prompt, 200, use_cache=False, seed=42)
assert torch.equal(ids_cached, ids_plain), "CACHE BUG: outputs diverged"
print("assert passed: 200 cached ids == 200 naive ids")
print("first 20 ids:", ids_cached[0, :20].tolist())
# Expected: assert passed: 200 cached ids == 200 naive ids
#           first 20 ids: <10 prompt ids + 10 generated, deterministic>

## 3. Benchmark: measure the speedup

Same 200-token generation, timed on both paths. On CPU expect 5–20×; on a T4 expect 2–5× for a model this small (kernel-launch overhead dominates at tiny sizes — Day 11 will quantify it).

In [ ]:
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Device:", device)

def timed(use_cache):
    p = prompt.to(device)
    t0 = time.perf_counter()
    generate(model, p, 200, use_cache=use_cache, seed=1)
    dt = time.perf_counter() - t0
    return dt, 200 / dt

dt_plain, tps_plain = timed(False)
dt_cached, tps_cached = timed(True)
print(f"non-cached: {dt_plain:6.2f} s  ({tps_plain:5.1f} tok/s)")
print(f"cached:     {dt_cached:6.2f} s  ({tps_cached:5.1f} tok/s)")
print(f"speedup:    {dt_plain / dt_cached:.1f}x")
# Expected (CPU, your numbers will vary):
#   Device: cpu
#   non-cached:  38.40 s  (  5.2 tok/s)
#   cached:       3.10 s  ( 64.5 tok/s)
#   speedup:    12.4x

## 4. The byte math: `kv_bytes()` and the serving table

bytes/token = 2 (K,V) × layers × KV-heads × head_dim × bytes/element. Assert the 8B/70B values, then print the serving-size table from the packet and check your numbers match.

In [ ]:
def kv_bytes(n_layers, n_kv_heads, head_dim, seq_len, batch=1, bytes_per=2):
    """KV cache bytes. Uses n_kv_heads (GQA), not n_heads."""
    return 2 * n_layers * n_kv_heads * head_dim * seq_len * batch * bytes_per


b8 = kv_bytes(32, 8, 128, 1)      # Llama-3.1-8B, per token, fp16
b70 = kv_bytes(80, 8, 128, 1)     # Llama-3.1-70B, per token, fp16
assert b8 == 131072, b8            # 128 KiB/token
assert b70 == 327680, b70           # 320 KiB/token
print(f"8B : {b8:,} B/token = {b8/1024:.0f} KiB/token")
print(f"70B: {b70:,} B/token = {b70/1024:.0f} KiB/token")

models = [("Llama-3.1-8B", 32, 8), ("Llama-3.1-70B", 80, 8)]
print(f"{'model':<14}{'seq':>6}{'b=1':>10}{'b=8':>10}")
for name, L, kv in models:
    for seq in (1024, 4096, 32768):
        g1 = kv_bytes(L, kv, 128, seq, 1) / 1024**3
        g8 = kv_bytes(L, kv, 128, seq, 8) / 1024**3
        print(f"{name:<14}{seq:>6}{g1:>9.2f} GiB{g8:>9.2f} GiB")
# Expected:
#   8B : 131,072 B/token = 128 KiB/token
#   70B: 327,680 B/token = 320 KiB/token
#   model            seq       b=1       b=8
#   Llama-3.1-8B    1024     0.12 GiB     1.00 GiB
#   Llama-3.1-8B    4096     0.50 GiB     4.00 GiB
#   Llama-3.1-8B   32768     4.00 GiB    32.00 GiB
#   Llama-3.1-70B   1024     0.31 GiB     2.50 GiB
#   Llama-3.1-70B   4096     1.25 GiB    10.00 GiB
#   Llama-3.1-70B  32768    10.00 GiB    80.00 GiB

## 5. Read the table like a serving engineer

- Bottom-right: 70B at 32k context, batch 8 = **80 GiB of cache**, on top of 140 GiB of weights. That is why a 70B does not fit on one 80 GB GPU.
- 8B at 32k/batch-8 = 32 GiB of cache alone — more than a T4's 16 GiB. In long-context serving, the *cache*, not the weights, is what kills you.
- Without GQA the 8B cache would be 4× larger (512 KiB/token). GQA was a serving win, not just a training trick.
- The cache removes *recomputation*, not *reads* — attention still scans the whole log each step. That growing memory traffic is why decode is memory-bandwidth-bound: tomorrow's topic.

**Done when:** the identical-ids assert passes, you have a measured speedup in your notes, and the serving table matches the packet. Tomorrow: prefill vs decode, arithmetic intensity — why batching is the entire economics of serving.

### What to try next

Change `n_kv` to 4 (full MHA) and re-run the `kv_bytes` asserts — watch the per-token bytes quadruple vs GQA. Then try `batch=4, seq=8192` on the 70B row and confirm the 10.24 GB checkpoint answer from the packet.